# Protein MLM with a Fresh BERT Model

This notebook builds a configurable masked language modeling workflow for protein sequences.

It is designed to:

- reuse the project's saved tokenizer assets
- create or reload tokenized dataset artifacts
- initialize a fresh `BertForMaskedLM` from a downloaded BERT config
- keep training optional until you explicitly enable it in the config block


## Imports

Keep imports isolated so dependency issues surface early and configuration stays uncluttered.


In [1]:
import hashlib
import inspect
import json
import math
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch

from tqdm.auto import tqdm
from transformers import BertConfig, BertForMaskedLM
from transformers import DataCollatorForLanguageModeling, PreTrainedTokenizerFast, Trainer, TrainingArguments, set_seed

from util.string_utils import humanizeNumericString
from util.file_utils import find_project_root, ensure_exists

## Configuration

All knobs for paths, tokenization reuse, model shape, and MLM training live here. Keep edits in this section instead of scattering flags across later cells.


In [ ]:
SEED = 42

# Tokenizer artifacts
TOKENIZATION_STRATEGY = "BPE"  # ["Unigram", "BPE", "WordPiece", "words", "pairs", "k-mers", "SentencePiece"]
VOCAB_SIZE = 512
RARE_RESIDUE_POLICY = "keep"

MAX_LENGTH = 512
TRAIN_FRACTION = 1.0
VALIDATION_SPLIT = 0.1
ENCODING_BATCH_SIZE = 512
SAMPLE_FILES = 5  # None

TOKENIZED_DATASET_FORMAT = "parquet"
STREAMING_SHUFFLE_BUFFER_SIZE = 10_000


# BERT Config
BASE_MODEL_CONFIG_ID = "bert-base-uncased"
HIDDEN_SIZE = 768
NUM_HIDDEN_LAYERS = 4
INTERMEDIATE_SIZE = 3072
MAX_POSITION_EMBEDDINGS = 512
TYPE_VOCAB_SIZE = 2
NUM_ATTENTION_HEADS = 12
HIDDEN_DROPOUT_PROB = 0.1
ATTENTION_PROBS_DROPOUT_PROB = 0.1

MLM_PROBABILITY = 0.15
NUM_TRAIN_EPOCHS = 1.0
MAX_STEPS = -1
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.0
LOGGING_STEPS = 50
EVAL_STEPS = 200
SAVE_STEPS = 200
SAVE_TOTAL_LIMIT = 2
DATALOADER_NUM_WORKERS = 0

EVALUATION_STRATEGY = "steps"
SAVE_STRATEGY = "steps"
LOAD_BEST_MODEL_AT_END = True
OVERWRITE_OUTPUT_DIR = True
USE_FP16 = torch.cuda.is_available()
USE_BF16 = False

TOKENIZER_ARTIFACTS_DIR = "results/protein_tokenization/artifacts/tokenizers"
MODEL_OUTPUT_DIR_RELATIVE_PATH = f"results/bert_mlm/{TOKENIZATION_STRATEGY.lower()}_vocab{VOCAB_SIZE}_{RARE_RESIDUE_POLICY}_max{MAX_LENGTH}"


# run options
REBUILD_TOKENIZED_DATASET = False
RESUME_FROM_CHECKPOINT = None
RUN_EVALUATION = False
RUN_TOKENIZATION = False  # When False, training requires an existing tokenized parquet dataset.
RUN_TRAINING = True

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
def bert_param_count(vocab_size, hidden_size, num_hidden_layers, intermediate_size, max_position_embeddings, type_vocab_size):
    # Embeddings
    embeddings = vocab_size * hidden_size + max_position_embeddings * hidden_size + type_vocab_size * hidden_size
    # Per layer
    per_layer = 4 * (hidden_size**2) + 2 * hidden_size * intermediate_size
    # Total
    total = embeddings + num_hidden_layers * per_layer
    return humanizeNumericString(total, decimals=0)

## Project Paths and Helpers

This section resolves the tokenizer artifact paths, fails fast when required assets are missing, and loads the saved fast tokenizer plus normalized corpus files.


In [ ]:
PROJECT_ROOT = find_project_root(Path.cwd())

TOKENIZER_ARTIFACTS_DIR = PROJECT_ROOT / TOKENIZER_ARTIFACTS_DIR
NORMALIZED_CORPUS_DIR = TOKENIZER_ARTIFACTS_DIR / "normalized_corpus"
TOKENIZER_RUN_DIR = TOKENIZER_ARTIFACTS_DIR / TOKENIZATION_STRATEGY / f"vocab{VOCAB_SIZE}_{RARE_RESIDUE_POLICY}" / "tokenized_corpus"

TOKENIZER_JSON_PATH = FAST_TOKENIZER_DIR / "tokenizer.json"
TOKENIZER_CONFIG_PATH = FAST_TOKENIZER_DIR / "tokenizer_config.json"
NORMALIZED_CORPUS_FILES = sorted(NORMALIZED_CORPUS_DIR.glob("*.txt"))
MODEL_OUTPUT_DIR = PROJECT_ROOT / MODEL_OUTPUT_DIR_RELATIVE_PATH
TOKENIZED_DATASET_DIR = MODEL_OUTPUT_DIR / TOKENIZED_DATASET_DIRNAME
TOKENIZED_DATASET_METADATA_PATH = TOKENIZED_DATASET_DIR / "metadata.json"

ensure_exists(TOKENIZER_ARTIFACTS_DIR, "Tokenizer artifacts directory")
ensure_exists(TOKENIZER_RUN_DIR, "Tokenizer run directory")
ensure_exists(FAST_TOKENIZER_DIR, "Fast tokenizer directory")
ensure_exists(TOKENIZER_JSON_PATH, "Fast tokenizer JSON")
ensure_exists(TOKENIZER_CONFIG_PATH, "Fast tokenizer config")
ensure_exists(NORMALIZED_CORPUS_DIR, "Normalized corpus directory")

if not NORMALIZED_CORPUS_FILES:
    raise FileNotFoundError(f"No normalized corpus text files found in: {NORMALIZED_CORPUS_DIR}")

if SAMPLE_FILES is not None:
    if SAMPLE_FILES <= 0:
        raise ValueError("SAMPLE_FILES must be None or a positive integer.")
    SELECTED_CORPUS_FILES = NORMALIZED_CORPUS_FILES[:SAMPLE_FILES]
else:
    SELECTED_CORPUS_FILES = NORMALIZED_CORPUS_FILES

if not SELECTED_CORPUS_FILES:
    raise ValueError("No normalized corpus files were selected. Increase SAMPLE_FILES or set it to None.")

set_seed(SEED)

tokenizer = PreTrainedTokenizerFast.from_pretrained(str(FAST_TOKENIZER_DIR))
tokenizer.model_max_length = MAX_LENGTH

missing_special_tokens = []
for token_name in ["pad_token", "unk_token", "cls_token", "sep_token", "mask_token"]:
    if getattr(tokenizer, token_name) is None:
        missing_special_tokens.append(token_name)
if missing_special_tokens:
    raise ValueError(f"Tokenizer is missing required special tokens: {missing_special_tokens}")

print(f"Loaded fast tokenizer from: {FAST_TOKENIZER_DIR}")
print(f"Found {len(NORMALIZED_CORPUS_FILES):,} normalized corpus files in: {NORMALIZED_CORPUS_DIR}")
print(f"Selected {len(SELECTED_CORPUS_FILES):,} normalized corpus files for this run.")

pd.DataFrame(
    [
        {"artifact": "tokenizer_run_dir", "path": str(TOKENIZER_RUN_DIR), "exists": TOKENIZER_RUN_DIR.exists()},
        {"artifact": "fast_tokenizer_dir", "path": str(FAST_TOKENIZER_DIR), "exists": FAST_TOKENIZER_DIR.exists()},
        {"artifact": "normalized_corpus_dir", "path": str(NORMALIZED_CORPUS_DIR), "exists": NORMALIZED_CORPUS_DIR.exists()},
        {"artifact": "normalized_corpus_files", "path": str(NORMALIZED_CORPUS_DIR), "exists": len(NORMALIZED_CORPUS_FILES) > 0, "count": len(NORMALIZED_CORPUS_FILES)},
        {"artifact": "selected_corpus_files", "path": str(NORMALIZED_CORPUS_DIR), "exists": len(SELECTED_CORPUS_FILES) > 0, "count": len(SELECTED_CORPUS_FILES)},
        {"artifact": "model_output_dir", "path": str(MODEL_OUTPUT_DIR), "exists": MODEL_OUTPUT_DIR.exists()},
        {"artifact": "tokenized_dataset_dir", "path": str(TOKENIZED_DATASET_DIR), "exists": TOKENIZED_DATASET_DIR.exists()},
        {"artifact": "tokenized_dataset_metadata", "path": str(TOKENIZED_DATASET_METADATA_PATH), "exists": TOKENIZED_DATASET_METADATA_PATH.exists()},
    ]
)

## Load or Build the Tokenized Parquet Dataset

This section expects an existing tokenized parquet cache for training. If you explicitly enable tokenization, it rebuilds the cache one normalized corpus file at a time so the notebook never keeps the full corpus in memory.


In [ ]:
def stable_fraction(*parts):
    digest = hashlib.blake2b("::".join(str(part) for part in parts).encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, byteorder="big") / float(1 << 64)


def get_tokenized_dataset_suffix(dataset_format):
    if dataset_format != "parquet":
        raise ValueError("TOKENIZED_DATASET_FORMAT must be 'parquet' for this pipeline.")
    return ".parquet"


def encode_batch_with_tokenizer(tokenizer_instance, text_batch):
    token_rows = [text.strip().split() for text in text_batch]
    encoded = tokenizer_instance(
        token_rows,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_special_tokens_mask=True,
    )

    batch_records = []
    encoded_dict = dict(encoded)
    batch_size = len(text_batch)
    for row_index in range(batch_size):
        record = {}
        for key, value in encoded_dict.items():
            record[key] = value[row_index]
        batch_records.append(record)

    return batch_records


def iter_list_batches(items, batch_size):
    for start_index in range(0, len(items), batch_size):
        yield items[start_index : start_index + batch_size]


def load_corpus_file_rows(file_path):
    file_rows = []
    with file_path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            normalized_text = line.strip()
            if not normalized_text:
                continue

            row_id = f"{file_path.name}:{line_number}"
            if stable_fraction("train_fraction", row_id) >= TRAIN_FRACTION:
                continue

            split_name = "train"
            if VALIDATION_SPLIT > 0 and stable_fraction("validation_split", row_id) < VALIDATION_SPLIT:
                split_name = "validation"

            file_rows.append((row_id, split_name, normalized_text))

    return file_rows


def group_file_rows_by_split(file_rows):
    rows_by_split = {"train": [], "validation": []}
    for row_id, split_name, normalized_text in file_rows:
        rows_by_split[split_name].append((row_id, normalized_text))
    return rows_by_split


def write_rows_to_parquet(rows, output_path):
    writer = None
    rows_written = 0

    try:
        for batch in iter_list_batches(rows, ENCODING_BATCH_SIZE):
            text_batch = [normalized_text for _, normalized_text in batch]
            tokenized_records = encode_batch_with_tokenizer(tokenizer, text_batch)

            rows_to_write = []
            for (row_id, _), tokenized_record in zip(batch, tokenized_records):
                record = dict(tokenized_record)
                record["row_id"] = row_id
                rows_to_write.append(record)

            if not rows_to_write:
                continue

            table = pa.Table.from_pylist(rows_to_write)
            if writer is None:
                writer = pq.ParquetWriter(
                    output_path,
                    table.schema,
                    compression=TOKENIZED_DATASET_PARQUET_COMPRESSION,
                )
            writer.write_table(table)
            rows_written += table.num_rows
    finally:
        if writer is not None:
            writer.close()

    if rows_written == 0 and output_path.exists():
        output_path.unlink()

    return rows_written


def build_tokenized_dataset_on_disk():
    if not 0 < TRAIN_FRACTION <= 1:
        raise ValueError("TRAIN_FRACTION must be in (0, 1].")
    if not 0 <= VALIDATION_SPLIT < 1:
        raise ValueError("VALIDATION_SPLIT must be in [0, 1).")
    if ENCODING_BATCH_SIZE <= 0:
        raise ValueError("ENCODING_BATCH_SIZE must be a positive integer.")
    if SAMPLE_FILES is not None and SAMPLE_FILES <= 0:
        raise ValueError("SAMPLE_FILES must be None or a positive integer.")

    dataset_format = TOKENIZED_DATASET_FORMAT
    get_tokenized_dataset_suffix(dataset_format)

    if not RUN_TOKENIZATION:
        raise FileNotFoundError(f"Expected an existing tokenized dataset at {TOKENIZED_DATASET_METADATA_PATH}, but RUN_TOKENIZATION is False.")

    if REBUILD_TOKENIZED_DATASET and TOKENIZED_DATASET_DIR.exists():
        shutil.rmtree(TOKENIZED_DATASET_DIR)
    elif TOKENIZED_DATASET_DIR.exists() and not TOKENIZED_DATASET_METADATA_PATH.exists():
        print(f"Removing incomplete tokenized dataset directory: {TOKENIZED_DATASET_DIR}")
        shutil.rmtree(TOKENIZED_DATASET_DIR)

    TOKENIZED_DATASET_DIR.mkdir(parents=True, exist_ok=True)

    split_dirs = {
        "train": TOKENIZED_DATASET_DIR / "train",
        "validation": TOKENIZED_DATASET_DIR / "validation",
    }
    for split_dir in split_dirs.values():
        split_dir.mkdir(parents=True, exist_ok=True)

    split_counts = {"train": 0, "validation": 0}
    data_files = {"train": [], "validation": []}
    data_file_rows = {"train": {}, "validation": {}}
    source_file_outputs = []

    file_progress = tqdm(SELECTED_CORPUS_FILES, desc="Tokenizing corpus files", unit="file")
    row_progress = tqdm(desc="Writing parquet rows", unit="row")

    try:
        for file_index, file_path in enumerate(file_progress):
            file_rows = load_corpus_file_rows(file_path)
            rows_by_split = group_file_rows_by_split(file_rows)

            source_file_entry = {
                "source_index": file_index,
                "source_file": str(file_path),
                "train_file": None,
                "train_rows": 0,
                "validation_file": None,
                "validation_rows": 0,
            }

            for split_name, rows in rows_by_split.items():
                if not rows:
                    continue

                output_path = split_dirs[split_name] / f"part-{file_index:05d}-{file_path.stem}.parquet"
                row_count = write_rows_to_parquet(rows, output_path)
                if row_count == 0:
                    continue

                output_path_str = str(output_path)
                data_files[split_name].append(output_path_str)
                data_file_rows[split_name][output_path_str] = row_count
                split_counts[split_name] += row_count
                source_file_entry[f"{split_name}_file"] = output_path_str
                source_file_entry[f"{split_name}_rows"] = row_count
                row_progress.update(row_count)

            source_file_outputs.append(source_file_entry)
            file_progress.set_postfix(train=split_counts["train"], validation=split_counts["validation"])
    except Exception:
        if TOKENIZED_DATASET_DIR.exists() and not TOKENIZED_DATASET_METADATA_PATH.exists():
            print(f"Cleaning failed tokenized dataset build at: {TOKENIZED_DATASET_DIR}")
            shutil.rmtree(TOKENIZED_DATASET_DIR, ignore_errors=True)
        raise
    finally:
        file_progress.close()
        row_progress.close()

    if split_counts["train"] == 0:
        raise ValueError("No normalized corpus rows were assigned to the training split.")

    metadata = {
        "format": dataset_format,
        "tokenized_dataset_dir": str(TOKENIZED_DATASET_DIR),
        "split_counts": split_counts,
        "data_files": data_files,
        "data_file_rows": data_file_rows,
        "source_file_outputs": source_file_outputs,
        "train_fraction": TRAIN_FRACTION,
        "validation_split": VALIDATION_SPLIT,
        "encoding_batch_size": ENCODING_BATCH_SIZE,
        "parquet_compression": TOKENIZED_DATASET_PARQUET_COMPRESSION,
        "max_length": MAX_LENGTH,
        "vocab_size": len(tokenizer),
        "sample_files": SAMPLE_FILES,
        "total_source_files": len(NORMALIZED_CORPUS_FILES),
        "selected_source_files": [str(path) for path in SELECTED_CORPUS_FILES],
    }

    TOKENIZED_DATASET_METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return metadata


def buffered_shuffle(iterator, buffer_size, seed):
    rng = np.random.default_rng(seed)
    buffer = []

    for item in iterator:
        buffer.append(item)
        if len(buffer) < buffer_size:
            continue

        index = int(rng.integers(0, len(buffer)))
        yield buffer.pop(index)

    while buffer:
        index = int(rng.integers(0, len(buffer)))
        yield buffer.pop(index)


def get_parquet_row_count(file_path):
    parquet_file = pq.ParquetFile(file_path)
    return parquet_file.metadata.num_rows


def validate_tokenized_dataset_metadata(metadata):
    if metadata.get("format") != "parquet":
        raise ValueError("The tokenized dataset must be stored as parquet for this notebook pipeline.")

    data_files = metadata.get("data_files", {})
    train_files = data_files.get("train", [])
    if not train_files:
        raise FileNotFoundError(f"No training parquet files were found in tokenized dataset metadata: {TOKENIZED_DATASET_METADATA_PATH}")

    for split_name, split_files in data_files.items():
        for file_path in split_files:
            if not Path(file_path).exists():
                raise FileNotFoundError(f"Missing tokenized {split_name} file: {file_path}")


def resolve_split_selection(split_name, metadata):
    split_counts = metadata.get("split_counts", {})
    source_file_outputs = metadata.get("source_file_outputs")

    if source_file_outputs:
        selected_entries = source_file_outputs if SAMPLE_FILES is None else source_file_outputs[:SAMPLE_FILES]
        selected_files = []
        selected_rows = 0
        available_files = 0

        for entry in source_file_outputs:
            if entry.get(f"{split_name}_file"):
                available_files += 1

        for entry in selected_entries:
            file_path = entry.get(f"{split_name}_file")
            row_count = int(entry.get(f"{split_name}_rows") or 0)
            if not file_path or row_count <= 0:
                continue
            selected_files.append(file_path)
            selected_rows += row_count

        return {
            "selected_files": selected_files,
            "selected_rows": selected_rows,
            "available_files": available_files,
            "available_rows": split_counts.get(split_name, 0),
        }

    all_files = metadata.get("data_files", {}).get(split_name, [])
    selected_files = all_files if SAMPLE_FILES is None else all_files[:SAMPLE_FILES]
    per_file_rows = metadata.get("data_file_rows", {}).get(split_name, {})
    selected_rows = 0
    for file_path in selected_files:
        row_count = per_file_rows.get(file_path)
        if row_count is None:
            row_count = get_parquet_row_count(Path(file_path))
        selected_rows += int(row_count)

    return {
        "selected_files": selected_files,
        "selected_rows": selected_rows,
        "available_files": len(all_files),
        "available_rows": split_counts.get(split_name, selected_rows),
    }


class DiskBackedTokenizedSplit(torch.utils.data.IterableDataset):
    def __init__(self, file_paths, row_count, shuffle=False, shuffle_buffer_size=0, seed=SEED):
        self.file_paths = [Path(file_path) for file_path in file_paths]
        self.row_count = row_count
        self.shuffle = shuffle
        self.shuffle_buffer_size = shuffle_buffer_size
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return self.row_count

    def _iter_records(self):
        for file_path in self.file_paths:
            parquet_file = pq.ParquetFile(file_path)
            for record_batch in parquet_file.iter_batches(batch_size=ENCODING_BATCH_SIZE):
                table = pa.Table.from_batches([record_batch])
                for record in table.to_pylist():
                    record.pop("row_id", None)
                    yield record

    def __iter__(self):
        iterator = self._iter_records()
        if self.shuffle and self.shuffle_buffer_size > 1 and self.row_count > 1:
            yield from buffered_shuffle(iterator, min(self.shuffle_buffer_size, self.row_count), self.seed + self.epoch)
            return

        yield from iterator


def load_tokenized_dataset_metadata():
    if TOKENIZED_DATASET_METADATA_PATH.exists() and not REBUILD_TOKENIZED_DATASET:
        print(f"Using existing tokenized dataset from: {TOKENIZED_DATASET_DIR}")
        metadata = json.loads(TOKENIZED_DATASET_METADATA_PATH.read_text(encoding="utf-8"))
        validate_tokenized_dataset_metadata(metadata)
        return metadata

    if not RUN_TOKENIZATION:
        raise FileNotFoundError(f"Expected an existing tokenized dataset metadata file at {TOKENIZED_DATASET_METADATA_PATH}. Set RUN_TOKENIZATION = True if you want to rebuild it file by file.")

    print(f"Building tokenized dataset at: {TOKENIZED_DATASET_DIR}")
    metadata = build_tokenized_dataset_on_disk()
    validate_tokenized_dataset_metadata(metadata)
    return metadata


def load_disk_backed_tokenized_dataset():
    metadata = load_tokenized_dataset_metadata()

    train_selection = resolve_split_selection("train", metadata)
    if not train_selection["selected_files"]:
        raise FileNotFoundError("No training parquet files were selected. Increase SAMPLE_FILES or rebuild the tokenized dataset.")

    dataset = {
        "train": DiskBackedTokenizedSplit(
            train_selection["selected_files"],
            train_selection["selected_rows"],
            shuffle=True,
            shuffle_buffer_size=STREAMING_SHUFFLE_BUFFER_SIZE,
            seed=SEED,
        )
    }
    split_selection = {"train": train_selection}

    validation_selection = resolve_split_selection("validation", metadata)
    if validation_selection["selected_files"] and validation_selection["selected_rows"] > 0:
        dataset["validation"] = DiskBackedTokenizedSplit(
            validation_selection["selected_files"],
            validation_selection["selected_rows"],
            shuffle=False,
            seed=SEED,
        )
        split_selection["validation"] = validation_selection

    return dataset, metadata, split_selection
  

def load_first_tokenized_record(split_name, split_selection):
    split_spec = split_selection.get(split_name)
    if split_spec is None or not split_spec["selected_files"]:
        raise ValueError(f"No files found for split: {split_name}")

    first_path = Path(split_spec["selected_files"][0])
    parquet_file = pq.ParquetFile(first_path)
    for record_batch in parquet_file.iter_batches(batch_size=1):
        table = pa.Table.from_batches([record_batch])
        records = table.to_pylist()
        if records:
            record = records[0]
            record.pop("row_id", None)
            return record

    raise ValueError(f"No records found for split: {split_name}")


tokenized_dataset, tokenized_dataset_metadata, tokenized_dataset_selection = load_disk_backed_token  ized_dataset()

pd.DataFrame(
    [
        {
            "split": split_name,
            "rows": tokenized_dataset_selection[split_name]["selected_rows"],
            "selected_files": len(tokenized_dataset_selection[split_name]["selected_files"]),
            "available_files": tokenized_dataset_selection[split_name]["available_files"],
            "available_rows": tokenized_dataset_selection[split_name]["available_rows"],
            "format": tokenized_dataset_metadata["format"],
        }
        for split_name in tokenized_dataset
    ]
)

In [ ]:
pd.Series(
    {
        "tokenized_dataset_dir": str(TOKENIZED_DATASET_DIR),
        "tokenized_dataset_metadata": str(TOKENIZED_DATASET_METADATA_PATH),
        "tokenized_dataset_format": TOKENIZED_DATASET_FORMAT,
        "parquet_compression": TOKENIZED_DATASET_PARQUET_COMPRESSION,
        "encoding_batch_size": ENCODING_BATCH_SIZE,
        "sample_files": SAMPLE_FILES,
        "selected_corpus_files": len(SELECTED_CORPUS_FILES),
        "run_tokenization": RUN_TOKENIZATION,
        "rebuild_tokenized_dataset": REBUILD_TOKENIZED_DATASET,
    }
)

## Inspect Tokenized Samples

Use this section to verify that the cached token IDs still map cleanly back to the saved tokenizer vocabulary.


In [ ]:
sample_split_name = "validation" if "validation" in tokenized_dataset else "train"
sample_record = load_first_tokenized_record(sample_split_name, tokenized_dataset_selection)
sample_tokens = tokenizer.convert_ids_to_tokens(sample_record["input_ids"])

pd.DataFrame(
    {
        "token_id": sample_record["input_ids"],
        "token": sample_tokens,
        "special_token_mask": sample_record["special_tokens_mask"],
    }
).head(32)

## Initialize a Fresh BERT MLM Model

This downloads the configuration template from Hugging Face and instantiates a new `BertForMaskedLM` with random weights sized to the local tokenizer vocabulary.


In [ ]:
def build_untrained_mlm_model():
    config = BertConfig.from_pretrained(BASE_MODEL_CONFIG_ID)

    overrides = {
        "hidden_size": MODEL_HIDDEN_SIZE,
        "num_hidden_layers": MODEL_NUM_HIDDEN_LAYERS,
        "num_attention_heads": MODEL_NUM_ATTENTION_HEADS,
        "intermediate_size": MODEL_INTERMEDIATE_SIZE,
    }
    for field_name, value in overrides.items():
        if value is not None:
            setattr(config, field_name, value)

    config.vocab_size = len(tokenizer)
    config.max_position_embeddings = max(config.max_position_embeddings, MAX_LENGTH + 2)
    config.hidden_dropout_prob = HIDDEN_DROPOUT_PROB
    config.attention_probs_dropout_prob = ATTENTION_PROBS_DROPOUT_PROB
    config.pad_token_id = tokenizer.pad_token_id
    config.bos_token_id = tokenizer.cls_token_id
    config.eos_token_id = tokenizer.sep_token_id

    if config.hidden_size % config.num_attention_heads != 0:
        raise ValueError("MODEL_HIDDEN_SIZE must be divisible by MODEL_NUM_ATTENTION_HEADS.")

    model = BertForMaskedLM(config)
    model.resize_token_embeddings(len(tokenizer))
    return model


model = build_untrained_mlm_model()
trainable_parameters = 0
for parameter in model.parameters():
    if parameter.requires_grad:
        trainable_parameters += parameter.numel()

pd.Series(
    {
        "config_source": BASE_MODEL_CONFIG_ID,
        "fresh_random_weights": True,
        "vocab_size": model.config.vocab_size,
        "max_position_embeddings": model.config.max_position_embeddings,
        "hidden_size": model.config.hidden_size,
        "num_hidden_layers": model.config.num_hidden_layers,
        "num_attention_heads": model.config.num_attention_heads,
        "trainable_parameters": trainable_parameters,
    }
)

## Configure the MLM Trainer

The trainer is assembled here, but training stays disabled unless you switch `RUN_TRAINING` on in the configuration block.


In [ ]:
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

run_evaluation = RUN_EVALUATION and "validation" in tokenized_dataset
save_strategy = SAVE_STRATEGY if RUN_TRAINING else "no"
eval_strategy = EVALUATION_STRATEGY if run_evaluation else "no"
load_best_model_at_end = LOAD_BEST_MODEL_AT_END and RUN_TRAINING and run_evaluation

if LOAD_BEST_MODEL_AT_END and not load_best_model_at_end:
    print("LOAD_BEST_MODEL_AT_END was disabled because evaluation is not active.")

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROBABILITY,
    pad_to_multiple_of=8 if USE_FP16 else None,
)

training_argument_values = {
    "output_dir": str(MODEL_OUTPUT_DIR),
    "overwrite_output_dir": OVERWRITE_OUTPUT_DIR,
    "do_train": RUN_TRAINING,
    "do_eval": run_evaluation,
    "eval_strategy": eval_strategy,
    "evaluation_strategy": eval_strategy,
    "save_strategy": save_strategy,
    "logging_strategy": "steps",
    "logging_steps": LOGGING_STEPS,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "max_steps": MAX_STEPS,
    "eval_steps": EVAL_STEPS if run_evaluation else None,
    "save_steps": SAVE_STEPS if RUN_TRAINING else None,
    "save_total_limit": SAVE_TOTAL_LIMIT,
    "load_best_model_at_end": load_best_model_at_end,
    "metric_for_best_model": "eval_loss" if load_best_model_at_end else None,
    "greater_is_better": False if load_best_model_at_end else None,
    "report_to": [],
    "remove_unused_columns": False,
    "dataloader_num_workers": DATALOADER_NUM_WORKERS,
    "seed": SEED,
    "fp16": USE_FP16,
    "bf16": USE_BF16,
}

training_argument_signature = inspect.signature(TrainingArguments.__init__).parameters
if "eval_strategy" in training_argument_signature:
    training_argument_values.pop("evaluation_strategy", None)
elif "evaluation_strategy" in training_argument_signature:
    training_argument_values.pop("eval_strategy", None)

supported_training_argument_values = {}
skipped_training_argument_names = []
for name, value in training_argument_values.items():
    if name not in training_argument_signature:
        skipped_training_argument_names.append(name)
        continue
    if value is None:
        continue
    supported_training_argument_values[name] = value

if skipped_training_argument_names:
    print(
        "Skipping unsupported TrainingArguments fields:",
        ", ".join(skipped_training_argument_names),
    )

training_args = TrainingArguments(**supported_training_argument_values)

trainer_values = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_dataset["train"],
    "eval_dataset": tokenized_dataset["validation"] if run_evaluation else None,
    "data_collator": data_collator,
    "processing_class": tokenizer,
    "tokenizer": tokenizer,
}
trainer_signature = inspect.signature(Trainer.__init__).parameters

if "processing_class" in trainer_signature:
    trainer_values.pop("tokenizer", None)
elif "tokenizer" in trainer_signature:
    trainer_values.pop("processing_class", None)

supported_trainer_values = {}
for name, value in trainer_values.items():
    if name not in trainer_signature:
        continue
    if value is None:
        continue
    supported_trainer_values[name] = value

trainer = Trainer(**supported_trainer_values)

pd.Series(
    {
        "run_training": RUN_TRAINING,
        "run_evaluation": run_evaluation,
        "load_best_model_at_end": load_best_model_at_end,
        "train_rows": tokenized_dataset_selection["train"]["selected_rows"],
        "validation_rows": tokenized_dataset_selection.get("validation", {}).get("selected_rows", 0),
        "mlm_probability": MLM_PROBABILITY,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "num_train_epochs": NUM_TRAIN_EPOCHS,
        "output_dir": str(MODEL_OUTPUT_DIR),
        "tokenized_dataset_dir": str(TOKENIZED_DATASET_DIR),
    }
)

## Optional Training and Evaluation

This cell is intentionally inert until `RUN_TRAINING` is set to `True`. It saves the trainer state, model weights, tokenizer copy, and evaluation metrics when enabled.


In [ ]:
if RUN_TRAINING:
    train_result = trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
    trainer.save_model()
    tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR / "tokenizer"))
    trainer.save_state()
    trainer.log_metrics("train", train_result.metrics)
    trainer.save_metrics("train", train_result.metrics)

    if run_evaluation:
        eval_metrics = trainer.evaluate()
        if "eval_loss" in eval_metrics:
            eval_metrics["eval_perplexity"] = math.exp(eval_metrics["eval_loss"])
        trainer.log_metrics("eval", eval_metrics)
        trainer.save_metrics("eval", eval_metrics)
        pd.Series(eval_metrics)
    else:
        pd.Series(train_result.metrics)
else:
    print("Trainer is configured but disabled.")
    print("Set RUN_TRAINING = True in the configuration cell to launch MLM training.")
    print(f"Fast tokenizer directory: {FAST_TOKENIZER_DIR}")
    print(f"Normalized corpus directory: {NORMALIZED_CORPUS_DIR}")
    print(f"Model output directory: {MODEL_OUTPUT_DIR}")

## Artifact and Run Summary

This section records the tokenizer artifact paths, normalized corpus source, and active training configuration so each run leaves a lightweight trail of what the notebook was set up to do.


In [ ]:
run_manifest = {
    "seed": SEED,
    "tokenization_strategy": TOKENIZATION_STRATEGY,
    "tokenizer_vocab_size": VOCAB_SIZE,
    "rare_residue_policy": RARE_RESIDUE_POLICY,
    "tokenizer_run_dir": str(TOKENIZER_RUN_DIR),
    "fast_tokenizer_dir": str(FAST_TOKENIZER_DIR),
    "normalized_corpus_dir": str(NORMALIZED_CORPUS_DIR),
    "normalized_corpus_files": len(NORMALIZED_CORPUS_FILES),
    "selected_corpus_files": len(SELECTED_CORPUS_FILES),
    "sample_files": SAMPLE_FILES,
    "model_output_dir": str(MODEL_OUTPUT_DIR),
    "tokenized_dataset_dir": str(TOKENIZED_DATASET_DIR),
    "tokenized_dataset_format": TOKENIZED_DATASET_FORMAT,
    "tokenized_dataset_parquet_compression": TOKENIZED_DATASET_PARQUET_COMPRESSION,
    "selected_train_tokenized_files": len(tokenized_dataset_selection["train"]["selected_files"]),
    "selected_validation_tokenized_files": len(tokenized_dataset_selection.get("validation", {}).get("selected_files", [])),
    "selected_train_rows": tokenized_dataset_selection["train"]["selected_rows"],
    "selected_validation_rows": tokenized_dataset_selection.get("validation", {}).get("selected_rows", 0),
    "base_model_config_id": BASE_MODEL_CONFIG_ID,
    "max_length": MAX_LENGTH,
    "validation_split": VALIDATION_SPLIT,
    "train_fraction": TRAIN_FRACTION,
    "encoding_batch_size": ENCODING_BATCH_SIZE,
    "run_tokenization": RUN_TOKENIZATION,
    "rebuild_tokenized_dataset": REBUILD_TOKENIZED_DATASET,
    "run_training": RUN_TRAINING,
    "run_evaluation": RUN_EVALUATION,
    "mlm_probability": MLM_PROBABILITY,
}

manifest_path = MODEL_OUTPUT_DIR / "run_config.json"
manifest_path.write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")

pd.DataFrame(
    [
        {
            "artifact": "tokenizer_run_dir",
            "path": str(TOKENIZER_RUN_DIR),
            "exists": TOKENIZER_RUN_DIR.exists(),
        },
        {
            "artifact": "fast_tokenizer_dir",
            "path": str(FAST_TOKENIZER_DIR),
            "exists": FAST_TOKENIZER_DIR.exists(),
        },
        {
            "artifact": "normalized_corpus_dir",
            "path": str(NORMALIZED_CORPUS_DIR),
            "exists": NORMALIZED_CORPUS_DIR.exists(),
        },
        {
            "artifact": "normalized_corpus_files",
            "path": str(NORMALIZED_CORPUS_DIR),
            "exists": len(NORMALIZED_CORPUS_FILES) > 0,
        },
        {
            "artifact": "tokenized_dataset_dir",
            "path": str(TOKENIZED_DATASET_DIR),
            "exists": TOKENIZED_DATASET_DIR.exists(),
        },
        {
            "artifact": "model_output_dir",
            "path": str(MODEL_OUTPUT_DIR),
            "exists": MODEL_OUTPUT_DIR.exists(),
        },
        {
            "artifact": "run_manifest",
            "path": str(manifest_path),
            "exists": manifest_path.exists(),
        },
    ]
)